In [1]:
#### Libraries ###
import os
import glob
import pandas as pd
import numpy as np
import scanpy as sc

import scipy.sparse as sp  
from scipy.sparse import csr_matrix
from scipy.io import mmread

import matplotlib.pyplot as plt

os.chdir("/data/scottaa/scrnaseq_pub")

In [22]:
mtx_file = "projects/E-MTAB-8559/raw_data/E-MTAB-8559_matrix.mtx"
genes_file = "projects/E-MTAB-8559/raw_data/E-MTAB-8559_genes.tsv"
cells_file = "projects/E-MTAB-8559/raw_data/E-MTAB-8559_barcodes.tsv"
# Load matrix
adata = sc.read_mtx(mtx_file).T

# Load genes (23,284 rows)
genes = pd.read_csv(
    genes_file,
    sep="\t",
    header=None
)

# Load cells (19,880 rows)
cells = pd.read_csv(cells_file, header=None)

# Assign correctly:
adata.var_names = genes[0].astype(str)   # genes to var
adata.obs_names = cells[0].astype(str)   # cells to obs

print("Genes assigned:", adata.var_names[:5])
print("Cells assigned:", adata.obs_names[:5])



#Fix gene names
gene_map = pd.read_csv("reference_data/ensembl_hgnc.csv")  # must have: ensembl_gene_id, hgnc_symbol

if "hgnc_symbol" in gene_map:

    ensg_to_hgnc = dict(zip(gene_map["ensembl_gene_id"], gene_map["hgnc_symbol"]))
    adata.var["ensembl_id"] = adata.var_names.astype(str)
    adata.var["gene_symbol"] = adata.var["ensembl_id"].map(ensg_to_hgnc)
    adata.var["gene_symbol"] = adata.var["gene_symbol"].fillna(adata.var["ensembl_id"])
    adata.var_names = adata.var["gene_symbol"].astype(str)

adata.var_names_make_unique()
adata.var.index.name = None
print("Genes names: ", adata.var_names[:10])



Genes assigned: Index(['ENSG00000000003', 'ENSG00000000419', 'ENSG00000000457',
       'ENSG00000000460', 'ENSG00000000938'],
      dtype='object')
Cells assigned: Index(['SAMEA6492740-AAACCCACAGTTAGGG', 'SAMEA6492740-AAACCCACATGTGTCA',
       'SAMEA6492740-AAACCCAGTCGCATGC', 'SAMEA6492740-AAACCCAGTCTTTCAT',
       'SAMEA6492740-AAACCCATCCGTGTCT'],
      dtype='object')
Genes names:  Index(['TSPAN6', 'DPM1', 'SCYL3', 'FIRRM', 'FGR', 'CFH', 'FUCA2', 'GCLC',
       'NFYA', 'STPG1'],
      dtype='object')


             donor_id sex  age FIGO_grade
sample                                   
SAMEA6492740      38b   F   81   stage 3c
SAMEA6492741       59   F   46   stage 3c
SAMEA6492742     74-1   F   73   stage 3b
SAMEA6492743       79   F   73   stage 3c
Series([], Name: count, dtype: int64)


In [29]:
adata.obs_names

Index(['SAMEA6492740-AAACCCACAGTTAGGG', 'SAMEA6492740-AAACCCACATGTGTCA',
       'SAMEA6492740-AAACCCAGTCGCATGC', 'SAMEA6492740-AAACCCAGTCTTTCAT',
       'SAMEA6492740-AAACCCATCCGTGTCT', 'SAMEA6492740-AAACCCATCCTCTCTT',
       'SAMEA6492740-AAACGAACAGCAGATG', 'SAMEA6492740-AAACGAATCCGGCTTT',
       'SAMEA6492740-AAACGAATCGAATGCT', 'SAMEA6492740-AAACGAATCGTCAAAC',
       ...
       'SAMEA6492743-TTTGGTTAGGCCTGAA', 'SAMEA6492743-TTTGGTTCAGAACATA',
       'SAMEA6492743-TTTGGTTGTCTGTGTA', 'SAMEA6492743-TTTGGTTGTTGCTGAT',
       'SAMEA6492743-TTTGGTTTCAGGACGA', 'SAMEA6492743-TTTGGTTTCATCGTAG',
       'SAMEA6492743-TTTGTTGCAACCGACC', 'SAMEA6492743-TTTGTTGGTCAGTCCG',
       'SAMEA6492743-TTTGTTGGTCCTGGTG', 'SAMEA6492743-TTTGTTGTCAGATTGC'],
      dtype='object', length=19880)